# Notebook 01 — Data Preparation
**Source:** `sales-dashboard/output/df_master.parquet`  
**Tujuan:** Agregasi dari level item → level order, siap untuk cohort & RFM.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Path relatif dari notebooks/ ke output proyek sebelumnya
SRC = Path('../../sales-dashboard/output/df_master.parquet')
OUT = Path('../output')
OUT.mkdir(exist_ok=True)

print('Source exists:', SRC.exists())

Source exists: True


## 1. Load df_master

In [2]:
df = pd.read_parquet(SRC)
df['order_month'] = df['order_month'].astype('period[M]')
print(f'df_master: {len(df):,} baris (level item)')
df.head(3)

df_master: 110,197 baris (level item)


,order_id,customer_id,product_id,seller_id,order_purchase_timestamp,order_month,order_year,order_quarter,price,freight_value,...,payment_type,payment_value,review_score,customer_state,customer_city,category,delivery_days,is_late,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-02 10:56:33,2017-10,2017,2017Q4,29.99,8.72,...,voucher,38.71,4.0,SP,sao paulo,housewares,8.0,0,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-24 20:41:37,2018-07,2018,2018Q3,118.70,22.76,...,boleto,141.46,4.0,BA,barreiras,perfumery,13.0,0,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-08 08:38:49,2018-08,2018,2018Q3,159.90,19.22,...,credit_card,179.12,5.0,GO,vianopolis,auto,9.0,0,2018-08-17 18:06:29,2018-09-04


## 2. Agregasi ke Level Order

In [3]:
df_orders = (
    df.groupby('order_id')
    .agg(
        customer_id    = ('customer_id', 'first'),
        customer_state = ('customer_state', 'first'),
        order_date     = ('order_purchase_timestamp', 'min'),
        order_revenue  = ('revenue', 'sum'),
        n_items        = ('product_id', 'count')
    )
    .reset_index()
)

df_orders['order_month'] = df_orders['order_date'].dt.to_period('M')

print(f'df_orders: {len(df_orders):,} order unik')
print(f'Customer unik: {df_orders["customer_id"].nunique():,}')
assert df_orders['order_id'].nunique() == len(df_orders), 'Ada duplikat order_id!'
print('Validasi: tidak ada duplikat order_id.')

df_orders: 96,478 order unik
Customer unik: 96,478
Validasi: tidak ada duplikat order_id.


## 3. Statistik Dasar

In [4]:
print('Periode data  :', df_orders['order_date'].min().date(),
      '→', df_orders['order_date'].max().date())
print('Total revenue : R$', f"{df_orders['order_revenue'].sum():,.0f}")
print('AOV (per order):', f"R$ {df_orders['order_revenue'].mean():,.2f}")
print('\nDistribusi orders per customer:')
orders_per_cust = df_orders.groupby('customer_id')['order_id'].count()
print(orders_per_cust.value_counts().head(5))

Periode data  : 2016-09-15 → 2018-08-29
Total revenue : R$ 15,419,774
AOV (per order): R$ 159.83

Distribusi orders per customer:
order_id
1    96478
Name: count, dtype: int64


## 4. Simpan ke Parquet

In [5]:
df_orders.to_parquet(OUT / 'df_orders.parquet', index=False)
print(f'Tersimpan: df_orders.parquet — {len(df_orders):,} baris, {df_orders.shape[1]} kolom')

Tersimpan: df_orders.parquet — 96,478 baris, 7 kolom
